# T30 — Prompt Registry & Versioning Lab

## Objective
Set up a centralized **Prompt Registry**. Version 5 distinct prompt iterations (`v1.0` through `v2.1`), evaluate performance, and demonstrate instant rollback capability when prompt regressions occur.

### Prompt Lifecycle & Registry Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      Central Prompt Registry                    │
├──────────────┬──────────────┬──────────────┬─────────────┬──────┤
│ Version 1.0  │ Version 1.1  │ Version 1.2  │ Version 2.0 │ v2.1 │
│ (Basic)      │ (Few-Shot)   │ (Chain-Thgt) │ (JSON Spec) │(Prod)│
└──────┬───────┴──────────────┴──────┬───────┴─────────────┴──┬───┘
       │                             │                        │
       │                      [ACTIVE PROMPT] ◄───────────────┘
       │                             │         Rollback Event
       └─────────────────────────────┴─────────────────────────┘
```



## 1. Environment Setup & Imports


In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")

client = OpenAI(api_key=api_key)
print("Environment initialized for Prompt Registry & Versioning!")


Environment initialized for Prompt Registry & Versioning!


## 2. Implement Centralized Prompt Registry Engine


In [2]:
class PromptRegistry:
    def __init__(self, registry_file: str = "prompt_registry.json"):
        self.registry_file = registry_file
        self.prompts = {}
        self.active_versions = {}
        self._load()

    def _load(self):
        if os.path.exists(self.registry_file):
            with open(self.registry_file, "r", encoding="utf-8") as f:
                data = json.load(f)
                self.prompts = data.get("prompts", {})
                self.active_versions = data.get("active_versions", {})

    def _save(self):
        with open(self.registry_file, "w", encoding="utf-8") as f:
            json.dump({
                "prompts": self.prompts,
                "active_versions": self.active_versions
            }, f, indent=2)

    def register_prompt(self, name: str, version: str, template: str, author: str, description: str):
        if name not in self.prompts:
            self.prompts[name] = {}
        self.prompts[name][version] = {
            "version": version,
            "template": template,
            "author": author,
            "description": description,
            "registered_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        # Automatically make latest registered version active if none set
        if name not in self.active_versions:
            self.active_versions[name] = version
        self._save()

    def set_active_version(self, name: str, version: str):
        if name in self.prompts and version in self.prompts[name]:
            self.active_versions[name] = version
            self._save()
            print(f"Switched active version of '{name}' to {version}")
        else:
            raise ValueError(f"Version {version} for '{name}' not found.")

    def rollback(self, name: str, target_version: str):
        print(f"ROLLBACK TRIGGERED: Rolling back '{name}' to {target_version}...")
        self.set_active_version(name, target_version)

    def get_prompt(self, name: str, version: str = None) -> dict:
        if not version:
            version = self.active_versions.get(name)
        return self.prompts.get(name, {}).get(version)

registry = PromptRegistry()
print("Prompt Registry Engine initialized!")


Prompt Registry Engine initialized!


## 3. Register 5 Prompt Versions for Customer Support RAG


In [3]:
prompt_name = "customer_support_rag"

v1_0 = "Answer the user question: {question} using context: {context}"

v1_1 = """You are a support agent.
Example Question: How to reset password? -> Answer: Click reset on login screen.

Context: {context}
Question: {question}
Answer:"""

v1_2 = """You are a senior support agent. Think step by step before answering.
Step 1: Check context for exact facts.
Step 2: Formulate concise response.

Context: {context}
Question: {question}"""

v2_0 = """Respond strictly in valid JSON object format with keys 'answer' and 'confidence_score'.

Context: {context}
Question: {question}"""

v2_1 = """[OVER-RESTRICTIVE REGRESSION VERSION] Reply in single word only. No explanations allowed.

Context: {context}
Question: {question}"""

# Register all 5 versions in Registry
registry.register_prompt(prompt_name, "v1.0", v1_0, "Intern", "Basic prompt baseline")
registry.register_prompt(prompt_name, "v1.1", v1_1, "Intern", "Added few-shot example")
registry.register_prompt(prompt_name, "v1.2", v1_2, "Senior Dev", "Chain-of-thought reasoning prompt")
registry.register_prompt(prompt_name, "v2.0", v2_0, "Lead Architect", "Strict JSON schema response format")
registry.register_prompt(prompt_name, "v2.1", v2_1, "Junior Dev", "Experimental over-restrictive single word prompt")

print("Successfully registered 5 prompt versions into registry!")


Successfully registered 5 prompt versions into registry!


## 4. Evaluate Versions & Demonstrate Rollback Capability


In [4]:
sample_context = "To request a refund, navigate to Account Settings > Billing > Request Refund. Refunds take 3-5 business days."
sample_question = "How do I request a refund and how long does it take?"

version_evaluations = []

for version in ["v1.0", "v1.1", "v1.2", "v2.0", "v2.1"]:
    prompt_obj = registry.get_prompt(prompt_name, version)
    formatted_prompt = prompt_obj["template"].format(context=sample_context, question=sample_question)
    
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": formatted_prompt}],
        temperature=0
    )
    
    answer = res.choices[0].message.content.strip()
    is_regression = len(answer.split()) < 3
    
    version_evaluations.append({
        "Version": version,
        "Author": prompt_obj["author"],
        "Description": prompt_obj["description"],
        "LLM Output Sample": answer[:65] + "...",
        "Status": "REGRESSION DETECTED" if is_regression else "PASS"
    })

df_prompts = pd.DataFrame(version_evaluations)
print("\n" + "="*80)
print("PROMPT REGISTRY VERSION EVALUATION SUMMARY")
print("="*80)
print(df_prompts.to_string(index=False))

print("\n" + "="*80)
registry.rollback(prompt_name, "v1.2")
active_now = registry.get_prompt(prompt_name)
print(f"Active Prompt is now version '{active_now['version']}' ({active_now['description']}).")
print("="*80)



PROMPT REGISTRY VERSION EVALUATION SUMMARY
Version         Author                                      Description                                                     LLM Output Sample              Status
   v1.0         Intern                            Basic prompt baseline  To request a refund, go to Account Settings, then select Billing,...                PASS
   v1.1         Intern                           Added few-shot example  To request a refund, navigate to Account Settings > Billing > Req...                PASS
   v1.2     Senior Dev                Chain-of-thought reasoning prompt  To request a refund, navigate to Account Settings > Billing > Req...                PASS
   v2.0 Lead Architect               Strict JSON schema response format {\n  "answer": "To request a refund, navigate to Account Settings ...                PASS
   v2.1     Junior Dev Experimental over-restrictive single word prompt                                                          Settings.... REGR

## 5. Conclusion & Deliverable Summary

In **Task 30 (Prompt Versioning)**:

1. **Centralized Prompt Registry**: Implemented `PromptRegistry` tracking prompt templates, authors, version numbers, and active production tags (`prompt_registry.json`).
2. **5 Prompt Versions Tracked**: Versioned 5 prompt iterations ranging from basic baseline (`v1.0`) to Chain-of-Thought (`v1.2`) and over-restrictive regression (`v2.1`).
3. **Rollback Capability**: Successfully detected prompt output regression in `v2.1` and performed instant zero-downtime rollback back to stable `v1.2`.

